# Fast-WDMamba V2: PSNR, FPS và GFLOPs trên nhiều bộ dữ liệu

Mặc định chọn **O-HAZY** và tiếp tục từ checkpoint Phase 3. Có thể đổi `DATASET_NAME` thành `reside6k`, `sots-its`, `sots-ots` hoặc `i-haze` ở cell cấu hình.
Kết quả cũ được giữ nguyên. Notebook này không chạy lại search hoặc train từ đầu.

**Bản v2.1 đã sửa lỗi deploy** `Inference tensors do not track version counter`.
Fusion tạo trọng số ngoài inference mode, CUDA Graph mặc định tắt. `RUN_MODE='auto'` tự dùng
checkpoint V2 đã lưu để benchmark lại sau lỗi; chưa có checkpoint thì chạy training bình thường.

- **Phục hồi PSNR:** fine-tune DEN trên crop 384, sau đó fine-tune toàn mạng trên **ảnh đầy đủ** ở max-side 768; GT MSE + 0,02 L1 + 0,1 KD MSE, EMA và early stopping bằng validation. Không giảm KD về 0 đột ngột.
- **Tăng FPS:** fusion Conv–BN và thử CUDA Graph ở FP32, chọn runtime nhanh hơn trên validation trước test. Không giảm độ phân giải hoặc đổi FP16 để so với FP32 cũ.
- **GFLOPs:** đếm Conv/Linear và phép chiếu SS2D; tách ước tính selective scan, FFT, Haar. Báo cáo 256×256 và từng kích thước test sau padding. `1 MAC = 2 FLOPs`; GFLOPs/ảnh khác GFLOP/s.

V2 giữ số block nên GFLOPs gần như giữ nguyên so Phase 3. CUDA Graph giảm overhead thực thi; không gọi đó là giảm độ phức tạp kiến trúc. **Chưa có kết quả GPU cho V2**: chỉ kết luận tăng cả PSNR và FPS khi báo cáo mới xác nhận.

## 1. Checkout và cấu hình

Dùng dataset đã có trong checkout VDT_GD2. Mặc định là `dataset/O-HAZY`; không tải lại dataset. Selector hợp lệ: `reside6k`, `sots-its` (SOTS Indoor/ITS), `sots-ots` (SOTS Outdoor/OTS), `o-hazy`, `i-haze`. SOTS chuẩn chỉ evaluation-only; muốn fine-tune phải đặt `TRAIN_ROOT` tới bộ train riêng và không được trỏ vào test.

Nếu chạy trong phiên Colab Phase 3, notebook tự tìm checkpoint cũ. Nếu mở một runtime mới, có thể upload checkpoint ở cell 3 hoặc đặt `PHASE3_CHECKPOINT` tới Drive. Chỉ checkpoint teacher được tự tải nếu chưa có.

In [ ]:
"""Configuration cell template; build_notebook_v2 injects DATASET_SPECS."""
from pathlib import Path
import json, os, platform, subprocess, sys, time

DATASET_NAME = 'o-hazy'  # @param ['reside6k', 'sots-its', 'sots-ots', 'o-hazy', 'i-haze']
DATA_ROOT_OVERRIDE = ''  # Thư mục bộ dữ liệu đã có; trống = đường dẫn trong checkout.
TRAIN_ROOT = ''          # Tập train ITS/OTS riêng nếu muốn fine-tune khi chọn SOTS.
EVAL_ONLY = None         # None = tự nhận diện; True = chỉ đánh giá; False = bắt buộc có train/val.
LOCAL_PROJECT = ''       # Checkout VDT_GD2 đã có trên Colab/Linux.
LOCAL_REPO = ''          # Thư mục chứa basicsr/archs/wavemamba_arch.py, nếu cần ghi đè.
CHECKPOINT = ''          # Teacher .pth riêng; trống = checkpoint mặc định bên dưới.
CHECKPOINT_ZIP = ''      # Archive teacher đã có nếu không muốn tải lại checkpoint.
PHASE3_CHECKPOINT = ''   # Student cùng dataset; O-HAZY tự tìm Phase 3, dataset khác không bắt buộc.
RUN_MODE = 'auto'       # @param ['auto', 'run', 'benchmark']
# auto: đã có checkpoint V2 thì chỉ benchmark; chưa có thì train rồi benchmark.
V2_CHECKPOINT = ''      # Khi benchmark: trống = checkpoint tốt nhất trong OUTPUT_ROOT.
TRY_CUDA_GRAPH = False # Tùy chọn thử nghiệm; mặc định dùng fusion eager để tránh lỗi capture.
EVAL_MAX_SIDE = 768      # 0 = ảnh gốc; cùng kích thước cho mọi model trong một lần chạy.
TRAIN_MAX_SIDE = 768
CROP_SIZE = 384          # Tự giới hạn theo cạnh ngắn của ảnh, làm tròn xuống bội 32.
MAX_CACHE_IMAGES = 40    # LRU trên RAM; vẫn lấy mẫu từ toàn bộ train của RESIDE-6K.

DATASET_SPECS = {'o-hazy': {'label': 'O-HAZY', 'relative_root': 'dataset/O-HAZY', 'kind': 'paired', 'can_train': True, 'checkpoint_name': 'Ohaze_1200_1600_27.22.pth', 'val_ids': [5, 15, 25, 35]}, 'i-haze': {'label': 'I-HAZE', 'relative_root': 'dataset/I-HAZE', 'kind': 'paired', 'can_train': True, 'checkpoint_name': 'Ohaze_1200_1600_27.22.pth', 'val_ids': []}, 'reside6k': {'label': 'RESIDE-6K', 'relative_root': 'dataset/RESIDE-6K', 'kind': 'paired', 'can_train': True, 'checkpoint_name': 'reside6k_32.15.pth', 'val_ids': []}, 'sots-its': {'label': 'SOTS Indoor / ITS', 'relative_root': 'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/indoor', 'kind': 'evaluation-only', 'can_train': False, 'checkpoint_name': 'reside6k_32.15.pth', 'val_ids': []}, 'sots-ots': {'label': 'SOTS Outdoor / OTS', 'relative_root': 'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/outdoor', 'kind': 'evaluation-only', 'can_train': False, 'checkpoint_name': 'reside6k_32.15.pth', 'val_ids': []}}
if DATASET_NAME not in DATASET_SPECS:
    raise ValueError(f'DATASET_NAME phải thuộc {list(DATASET_SPECS)}')
SPEC = DATASET_SPECS[DATASET_NAME]
REPO_URL = 'https://github.com/Darkin72/WDMamba.git'
PROJECT_URL = 'https://github.com/Darkin72/VDT_GD2.git'
PROJECT_ROOT = Path(LOCAL_PROJECT).expanduser().resolve() if LOCAL_PROJECT else next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p/'dataset').is_dir()), Path('/content/VDT_GD2'))
WORK_ROOT = Path('/content/wdmamba_v2') / DATASET_NAME
OUTPUT_ROOT = WORK_ROOT / 'results' / DATASET_NAME
ENV_ROOT = Path('/content/wdmamba_o_hazy_env311')
PYTHON = ENV_ROOT / 'bin/python'
if platform.system() != 'Linux':
    raise RuntimeError('Chạy notebook trên Colab/Linux GPU. Kiểm thử CPU dùng test_cpu.py.')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def run_command(args, cwd=None, log_name=None):
    args = list(map(str, args))
    print('$', ' '.join(args), flush=True)
    with (WORK_ROOT / (log_name or 'setup.log')).open('a', encoding='utf-8') as log:
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as proc:
            try:
                for line in proc.stdout:
                    print(line, end='', flush=True)
                    log.write(line); log.flush()
                rc = proc.wait()
            except BaseException:
                proc.kill(); proc.wait(); raise
    if rc:
        raise subprocess.CalledProcessError(rc, args)


if not DATA_ROOT_OVERRIDE and not PROJECT_ROOT.exists():
    run_command(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', PROJECT_URL, PROJECT_ROOT])
    # --skip-checks supports the SOTS directory name containing brackets.
    run_command(['git', 'sparse-checkout', 'set', '--skip-checks', SPEC['relative_root'],
                 'solution/Haze Clear'], cwd=PROJECT_ROOT)
DATA_ROOT = Path(DATA_ROOT_OVERRIDE).expanduser().resolve() if DATA_ROOT_OVERRIDE else PROJECT_ROOT / SPEC['relative_root']
if not DATA_ROOT.is_dir() and not DATA_ROOT_OVERRIDE and (PROJECT_ROOT/'.git/info/sparse-checkout').is_file():
    run_command(['git', 'sparse-checkout', 'add', '--skip-checks', SPEC['relative_root']], cwd=PROJECT_ROOT)
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Không có {SPEC["label"]} tại {DATA_ROOT}. Đặt DATA_ROOT_OVERRIDE tới dữ liệu có sẵn.')
project_arch = PROJECT_ROOT / 'solution/Haze Clear'
REPO_ROOT = (Path(LOCAL_REPO).expanduser().resolve() if LOCAL_REPO else
             project_arch if (project_arch/'basicsr/archs/wavemamba_arch.py').is_file() else Path('/content/WDMamba_v2'))
if not REPO_ROOT.exists():
    run_command(['git', 'clone', REPO_URL, REPO_ROOT])
if not (REPO_ROOT/'basicsr/archs/wavemamba_arch.py').is_file():
    raise FileNotFoundError('LOCAL_REPO phải chứa basicsr/archs/wavemamba_arch.py.')

CONFIG = {
    'dataset_name': DATASET_NAME, 'eval_only': EVAL_ONLY, 'train_root': TRAIN_ROOT,
    'repo_root': str(REPO_ROOT), 'project_root': str(PROJECT_ROOT),
    'work_root': str(WORK_ROOT), 'output_root': str(OUTPUT_ROOT), 'data_root': str(DATA_ROOT),
    'checkpoint': CHECKPOINT, 'checkpoint_zip': CHECKPOINT_ZIP, 'checkpoint_name': SPEC['checkpoint_name'],
    'checkpoint_drive_id': '1Vm-1mtMSc9YIUICq4nWKu-4AHCtYnPpF',
    'phase3_checkpoint': PHASE3_CHECKPOINT, 'v2_checkpoint': V2_CHECKPOINT, 'val_ids': SPEC['val_ids'], 'seed': 2026,
    'den_blocks': 1, 'eval_max_side': EVAL_MAX_SIDE, 'train_max_side': TRAIN_MAX_SIDE,
    'crop_size': CROP_SIZE, 'max_cache_images': MAX_CACHE_IMAGES,
    'crop_steps': 1000, 'full_steps': 600, 'accum_steps': 2,
    'crop_lr': 2e-5, 'full_den_lr': 5e-6, 'full_lfrn_lr': 1e-6,
    'l1_weight': 0.02, 'kd_mse_weight': 0.10, 'weight_decay': 1e-5, 'ema_decay': 0.99,
    'val_every': 100, 'log_every': 50, 'patience': 4, 'crop_border': 4,
    'warmup': 5, 'timing_repeats': 30, 'psnr_tolerance_db': 0.30,
    'try_cuda_graph': TRY_CUDA_GRAPH, 'fusion_max_error': 1e-4,
    'min_psnr_gain_db': 0.10, 'min_fps_ratio': 1.03, 'deploy_psnr_tolerance_db': 0.01,
}
if DATASET_NAME == 'o-hazy' and not CHECKPOINT and not CHECKPOINT_ZIP:
    prior_inputs = Path('/content/wdmamba_o_hazy_final/inputs.json')
    if prior_inputs.is_file():
        prior = json.loads(prior_inputs.read_text(encoding='utf-8'))
        if Path(prior['checkpoint']).is_file():
            CONFIG['checkpoint'] = prior['checkpoint']
CONFIG_PATH = WORK_ROOT / 'config.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')
print('Dataset:', SPEC['label'], '| thư mục:', DATA_ROOT)
print('Chế độ:', 'SOTS chỉ đánh giá mặc định' if DATASET_NAME.startswith('sots-') and not TRAIN_ROOT else 'tự nhận diện train/val/test')
if DATASET_NAME == 'i-haze' and not CHECKPOINT:
    print('I-HAZE dùng teacher O-HAZY để chuyển miền; có thể đặt CHECKPOINT riêng.')
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


## 2. Môi trường GPU riêng

Tái sử dụng môi trường Python 3.11/CUDA từ Phase 3 nếu có. Cell kiểm tra CUDA/Mamba trước train. Tất cả kết quả chính là FP32 và TF32 tắt. Phần tính GFLOPs dùng hook phân tích trực tiếp, không cần cài THOP và không để lại hook/buffer profiling trên model.

In [ ]:
run_command(['nvidia-smi'])
run_command(['nvcc', '--version'])
run_command([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
run_command([sys.executable, '-m', 'uv', 'python', 'install', '3.11'])
if not PYTHON.is_file():
    run_command([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', ENV_ROOT])
run_command([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
             '--index-url', 'https://download.pytorch.org/whl/cu128', 'torch==2.8.0', 'torchvision==0.23.0'])
constraints = ENV_ROOT / 'constraints.txt'
constraints.write_text('torch==2.8.0\ntorchvision==0.23.0\nnumpy<2\nsetuptools<81\n', encoding='utf-8')
run_command([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON, '--constraint', constraints,
             'numpy<2', 'opencv-python-headless<4.12', 'scipy', 'scikit-image', 'einops',
             'timm==1.0.15', 'pytorch-wavelets', 'PyWavelets', 'setuptools<81', 'packaging',
             'ninja', 'wheel', 'transformers', 'gdown', 'tqdm'])
run_command([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON, '--constraint', constraints,
             '--no-build-isolation', 'causal-conv1d', 'mamba-ssm'])
freeze = subprocess.check_output([sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', str(PYTHON)], text=True)
(OUTPUT_ROOT / 'environment.txt').write_text(freeze, encoding='utf-8')
check = r"""
import torch
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
assert torch.cuda.is_available()
u = torch.zeros(1,4,16,device='cuda')
y = selective_scan_fn(u, torch.ones_like(u), -torch.ones(4,1,device='cuda'),
                      torch.ones(1,1,1,16,device='cuda'), torch.ones(1,1,1,16,device='cuda'), delta_softplus=True)
assert torch.isfinite(y).all()
print('CUDA/Mamba PASS:', torch.__version__, torch.version.cuda, torch.cuda.get_device_name())
"""
run_command([PYTHON, '-c', check])

## 3. Checkpoint student

O-HAZY tiếp tục checkpoint Phase 3 đã có; nếu thiếu, cell cho phép upload. Các dataset khác để trống `PHASE3_CHECKPOINT` để khởi tạo bằng nén teacher, hoặc chỉ định student đã train đúng dataset. Với SOTS chỉ đánh giá, student tùy chọn phải thuộc RESIDE-6K. Teacher đặt ở `CHECKPOINT`, không đặt vào `PHASE3_CHECKPOINT`.

In [ ]:
if DATASET_NAME == 'o-hazy' and not CONFIG['phase3_checkpoint']:
    candidates = [
        PROJECT_ROOT/'solution/Haze Clear/phase 3/fast_wdmamba_o_hazy_best.pth',
        Path('/content/wdmamba_o_hazy_final/results/fast_wdmamba_o_hazy_best.pth'),
        Path('/content/fast_wdmamba_o_hazy_best.pth'),
    ]
    found = next((p for p in candidates if p.is_file()), None)
    if found is None:
        from google.colab import files
        uploaded = files.upload()
        names = [name for name in uploaded if name.endswith('.pth')]
        if len(names) != 1:
            raise ValueError('Chọn đúng một checkpoint Phase 3 .pth.')
        found = Path(names[0]).resolve()
    CONFIG['phase3_checkpoint'] = str(found)
if CONFIG['phase3_checkpoint'] and not Path(CONFIG['phase3_checkpoint']).is_file():
    raise FileNotFoundError(CONFIG['phase3_checkpoint'])
print('Student checkpoint:', CONFIG['phase3_checkpoint'] or 'nén teacher; không tự nạp checkpoint O-HAZY')

## 4. Mã chạy tự chứa

Notebook chứa toàn bộ mã core Phase 3 và runner V2. Teacher, checkpoint Phase 3 và dữ liệu được kiểm tra SHA256 trước train. Chỉ train được dùng cho cache/KD; validation chọn checkpoint và runtime, test chỉ dùng lập bảng cuối.

Fine-tuning dùng ảnh đầy đủ trong bước joint để khớp ngữ cảnh Mamba/FFT lúc đánh giá. Checkpoint ban đầu cũng được chấm validation và lưu; checkpoint tệ hơn trên validation không ghi đè bản tốt nhất. Các checkpoint lưu là **trọng số eager chưa fusion** để nạp lại đúng kiến trúc.

In [ ]:
CORE_SOURCE = r'''
"""Shared paired-data, architecture and metrics utilities for Fast-WDMamba V2."""
from pathlib import Path
import copy
import csv
import hashlib
import importlib.machinery
import importlib.util
import json
import math
import random
import re
import subprocess
import sys
import time
import zipfile

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from skimage.metrics import structural_similarity

EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}


def dump_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def write_csv(path, rows):
    if not rows:
        return
    with Path(path).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        if z.testzip() is not None:
            raise ValueError(f'ZIP lỗi: {archive}')
        for item in z.infolist():
            if not (destination / item.filename).resolve().is_relative_to(destination):
                raise ValueError('Đường dẫn ZIP nằm ngoài thư mục giải nén.')
        z.extractall(destination)


DATASET_SPECS = {
    'o-hazy': {'label': 'O-HAZY', 'relative_root': 'dataset/O-HAZY', 'kind': 'paired', 'can_train': True,
               'checkpoint_name': 'Ohaze_1200_1600_27.22.pth', 'val_ids': [5, 15, 25, 35]},
    'i-haze': {'label': 'I-HAZE', 'relative_root': 'dataset/I-HAZE', 'kind': 'paired', 'can_train': True,
               'checkpoint_name': 'Ohaze_1200_1600_27.22.pth', 'val_ids': []},
    'reside6k': {'label': 'RESIDE-6K', 'relative_root': 'dataset/RESIDE-6K', 'kind': 'paired', 'can_train': True,
                 'checkpoint_name': 'reside6k_32.15.pth', 'val_ids': []},
    'sots-its': {'label': 'SOTS Indoor / ITS', 'relative_root': 'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/indoor',
                 'kind': 'evaluation-only', 'can_train': False, 'checkpoint_name': 'reside6k_32.15.pth', 'val_ids': []},
    'sots-ots': {'label': 'SOTS Outdoor / OTS', 'relative_root': 'dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]/outdoor',
                 'kind': 'evaluation-only', 'can_train': False, 'checkpoint_name': 'reside6k_32.15.pth', 'val_ids': []},
}


def normalize_dataset_name(name):
    key = str(name or 'o-hazy').strip().lower().replace('_', '-').replace('/', '-').replace(' ', '-')
    key = re.sub(r'-+', '-', key).strip('-')
    aliases = {'o-haze': 'o-hazy', 'ohazy': 'o-hazy', 'ihaze': 'i-haze', 'sots-indoor': 'sots-its', 'sots-indoor-its': 'sots-its',
               'its': 'sots-its', 'sots-outdoor': 'sots-ots', 'sots-outdoor-ots': 'sots-ots', 'ots': 'sots-ots', 'reside-6k': 'reside6k'}
    key = aliases.get(key, key)
    if key not in DATASET_SPECS:
        raise ValueError(f'DATASET_NAME không hợp lệ: {name}. Chọn một trong {list(DATASET_SPECS)}')
    return key


def _kind(path, root):
    parents = {a.lower() for a in Path(path).relative_to(root).parts[:-1]}
    stem = Path(path).stem.lower()
    if parents & {'gt', 'clear', 'clean', 'groundtruth', 'ground_truth'} or stem.endswith(('_gt', '_clear', '_clean')):
        return 'clear'
    if parents & {'hazy', 'haze', 'input', 'inputs', 'hazy_images'} or stem.endswith(('_hazy', '_haze', '_input')):
        return 'hazy'
    return None


def _base_stem(path):
    stem = Path(path).stem.lower()
    stem = re.sub(r'(?i)([_-]?hazy|[_-]?haze|[_-]?gt|[_-]?clear|[_-]?clean)$', '', stem)
    return re.sub(r'\d+', lambda m: str(int(m[0])), stem)


def collect_pairs(root, dataset_name='o-hazy'):
    """Pair clear/hazy images across explicit split folders and common suffix conventions."""
    dataset_name = normalize_dataset_name(dataset_name)
    root = Path(root).resolve()
    groups = {}
    for p in sorted(root.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in EXT:
            continue
        kind = _kind(p, root)
        if kind is None:
            continue
        rel_parts = p.relative_to(root).parts
        split = next((x.lower() for x in rel_parts if x.lower() in {'train', 'val', 'validation', 'test'}), None)
        if split == 'validation': split = 'val'
        groups.setdefault(split or 'all', {'clear': [], 'hazy': []})[kind].append(p.resolve())
    pairs = []
    for split, sides in sorted(groups.items()):
        exact, normalized = {}, {}
        for clear in sides['clear']:
            for table, key in ((exact, clear.stem.lower()), (normalized, _base_stem(clear))):
                if key in table:
                    raise ValueError(f'Trùng ảnh clear trong split {split}: {table[key]} và {clear}')
                table[key] = clear
        used, ids = set(), set()
        for hazy in sides['hazy']:
            clear = exact.get(hazy.stem.lower()) or normalized.get(_base_stem(hazy))
            # RESIDE: one clear scene can have several atmospheric variants.
            if clear is None and dataset_name in {'reside6k', 'sots-its', 'sots-ots'}:
                clear = normalized.get(_base_stem(Path(hazy.stem.split('_')[0])))
            if clear is None:
                raise ValueError(f'Không ghép được ảnh hazy với clear trong split {split}: {hazy}')
            sid = hazy.stem.lower()
            if sid in ids:
                raise ValueError(f'Trùng ID hazy trong split {split}: {sid}')
            ids.add(sid); used.add(clear)
            pairs.append({'id': sid, 'scene': _base_stem(clear), 'hazy': str(hazy), 'clear': str(clear),
                          'source_split': None if split == 'all' else split})
        if set(sides['clear']) != used:
            raise ValueError(f'Có ảnh clear thiếu hazy trong split {split}: {set(sides["clear"])-used}')
    if not pairs:
        raise ValueError(f'Không tìm thấy cặp clear/hazy trong {root}')
    if dataset_name == 'o-hazy':
        # Phase 3 protocol: IDs 1-40 for tuning, 41-45 held out for test.
        def numeric(row):
            m = re.match(r'0*(\d+)', row['id'])
            return int(m.group(1)) if m else None
        by_id = {}
        for row in pairs:
            sid = numeric(row)
            if sid in by_id or sid is None:
                raise ValueError(f'Trùng hoặc thiếu ID scene O-HAZY: {row}')
            by_id[sid] = row
        if set(by_id) != set(range(1, 46)):
            raise ValueError('O-HAZY cần đủ 45 scene 01–45 để giữ protocol Phase 3.')
        pairs = [by_id[i] | {'id': i} for i in sorted(by_id)]
    return pairs


def _holdout_train(rows, seed, fraction):
    scenes = sorted({r['scene'] for r in rows})
    if len(scenes) < 2:
        raise ValueError('Cần ít nhất hai scene train để tách validation.')
    random.Random(seed).shuffle(scenes)
    val = set(scenes[:min(len(scenes)-1, max(1, round(len(scenes)*fraction)))])
    return [r for r in rows if r['scene'] not in val], [r for r in rows if r['scene'] in val]


def partition(pairs, cfg):
    dataset_name = normalize_dataset_name(cfg.get('dataset_name', 'o-hazy'))
    if dataset_name == 'o-hazy' and all(isinstance(p['id'], int) for p in pairs):
        val = set(cfg.get('val_ids', [5, 15, 25, 35]))
        if not val or not val <= set(range(1, 41)):
            raise ValueError('val_ids O-HAZY phải thuộc 1–40 và không rỗng.')
        result = {'train': [p for p in pairs if p['id'] <= 40 and p['id'] not in val],
                  'val': [p for p in pairs if p['id'] in val],
                  'test': [p for p in pairs if 41 <= p['id'] <= 45]}
    else:
        explicit = {p.get('source_split') for p in pairs}
        if None in explicit and len(explicit) > 1:
            raise ValueError('Không trộn dữ liệu phẳng với thư mục train/val/test.')
        if dataset_name.startswith('sots-'):
            # Some repackaged SOTS have train/val folders. They are still benchmark test images.
            result = {'train': [], 'val': [], 'test': list(pairs)}
        elif explicit == {None}:
            raise ValueError('Dataset train cần thư mục train và test; không tự lấy test làm validation.')
        else:
            result = {k: [p for p in pairs if p.get('source_split') == k] for k in ('train', 'val', 'test')}
            if result['train'] and not result['val']:
                result['train'], result['val'] = _holdout_train(result['train'], cfg.get('seed', 2026), .1)
    assert_disjoint(result)
    return result


def assert_disjoint(splits):
    seen = {}
    for split, rows in splits.items():
        ids = set()
        for row in rows:
            if row['id'] in ids:
                raise ValueError(f'Trùng ID trong split {split}: {row["id"]}')
            ids.add(row['id'])
            keys = [('scene', row['scene']), ('path', row['clear']), ('path', row['hazy'])]
            keys += [('clear_sha256', row['clear_sha256'])] if row.get('clear_sha256') else []
            keys += [('hazy_sha256', row['hazy_sha256'])] if row.get('hazy_sha256') else []
            for key in keys:
                if key in seen and seen[key] != split:
                    raise ValueError(f'Rò rỉ dữ liệu giữa {seen[key]} và {split}: {key}')
                seen[key] = split


def dataset_splits(cfg):
    name = normalize_dataset_name(cfg.get('dataset_name', 'o-hazy'))
    data = Path(cfg.get('data_root') or Path(cfg['project_root']) / DATASET_SPECS[name]['relative_root']).expanduser().resolve()
    if not data.is_dir():
        raise FileNotFoundError(f'Không có dataset {name} tại {data}. Đặt DATA_ROOT_OVERRIDE tới dữ liệu có sẵn.')
    splits = partition(collect_pairs(data, name), {**cfg, 'dataset_name': name})
    if cfg.get('train_root'):
        if not name.startswith('sots-'):
            raise ValueError('TRAIN_ROOT chỉ dành cho train ITS/OTS riêng khi đánh giá trên SOTS.')
        train_root = Path(cfg['train_root']).expanduser().resolve()
        if train_root == data or train_root.is_relative_to(data) or data.is_relative_to(train_root):
            raise ValueError('TRAIN_ROOT phải tách biệt với thư mục benchmark SOTS.')
        training = collect_pairs(train_root, name)
        explicit = {r.get('source_split') for r in training}
        if explicit == {None}:
            splits['train'], splits['val'] = _holdout_train(training, cfg.get('seed', 2026), .1)
        elif explicit <= {'train', 'val'} and 'train' in explicit:
            splits['train'] = [r for r in training if r['source_split'] == 'train']
            splits['val'] = [r for r in training if r['source_split'] == 'val']
            if not splits['val']:
                splits['train'], splits['val'] = _holdout_train(splits['train'], cfg.get('seed', 2026), .1)
        else:
            raise ValueError('TRAIN_ROOT chỉ chứa train/val hoặc các cặp ảnh phẳng, không chứa test.')
    requested = cfg.get('eval_only')
    eval_only = (not splits['train']) if requested is None else bool(requested)
    if not splits['test'] or (not eval_only and (not splits['train'] or not splits['val'])):
        raise ValueError('Cần test để đánh giá; fine-tune còn cần train và validation riêng. SOTS cần TRAIN_ROOT để train.')
    assert_disjoint(splits)
    return data, splits, eval_only


def prepare_inputs(cfg):
    work = Path(cfg['work_root']); work.mkdir(parents=True, exist_ok=True)
    dataset_name = normalize_dataset_name(cfg.get('dataset_name', 'o-hazy'))
    spec = DATASET_SPECS[dataset_name]
    data, splits, eval_only = dataset_splits(cfg)
    for split, rows in splits.items():
        for row in rows:
            row['dataset_name'] = dataset_name; row['split'] = split
            row['hazy_sha256'] = sha256(row['hazy']); row['clear_sha256'] = sha256(row['clear'])
    assert_disjoint(splits)
    checkpoint = Path(cfg['checkpoint']).expanduser().resolve() if cfg.get('checkpoint') else None
    if checkpoint is None:
        root = work / 'weights'; matches = list(root.rglob(cfg.get('checkpoint_name', spec['checkpoint_name']))) if root.is_dir() else []
        if not matches:
            import gdown
            archive = Path(cfg.get('checkpoint_zip') or (work / 'WDMamba_ckpts.zip'))
            if not archive.is_file():
                partial = archive.with_suffix('.zip.partial')
                result = gdown.download(id=cfg['checkpoint_drive_id'], output=str(partial), quiet=False, use_cookies=False, resume=True)
                if result is None or not zipfile.is_zipfile(partial): raise RuntimeError('Không tải được checkpoint.')
                partial.replace(archive)
            safe_extract(archive, root); matches = list(root.rglob(cfg.get('checkpoint_name', spec['checkpoint_name'])))
        if len(matches) != 1: raise FileNotFoundError(f'Cần đúng một checkpoint {cfg.get("checkpoint_name", spec["checkpoint_name"])}; tìm thấy {matches}')
        checkpoint = matches[0].resolve()
    if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
    inputs = {'dataset_name': dataset_name, 'data_root': str(data), 'checkpoint': str(checkpoint),
              'checkpoint_sha256': sha256(checkpoint), 'splits': splits, 'evaluation_only': eval_only,
              'train_root': str(Path(cfg['train_root']).expanduser().resolve()) if cfg.get('train_root') else ''}
    dump_json(work / 'inputs.json', inputs)
    write_csv(work / 'split_manifest.csv', [p for rows in splits.values() for p in rows])
    print(f'Dataset {spec["label"]}:', {k: len(v) for k, v in splits.items()}, flush=True)
    print('Checkpoint:', checkpoint, flush=True)
    return inputs

def import_architecture(root):
    # Avoid BasicSR __init__, which imports unrelated training stacks.
    for name, rel in [('basicsr', 'basicsr'), ('basicsr.utils', 'basicsr/utils'),
                      ('basicsr.archs', 'basicsr/archs')]:
        if name in sys.modules:
            raise RuntimeError(f'Runner cần process sạch; đã import {name}.')
        spec = importlib.machinery.ModuleSpec(name, loader=None, is_package=True)
        spec.submodule_search_locations = [str(Path(root) / rel)]
        sys.modules[name] = importlib.util.module_from_spec(spec)
    from basicsr.archs.wavemamba_arch import WaveMamba
    from basicsr.archs.detail_enhance_net import DENet
    from basicsr.archs.wavelet import DWT, IWT
    return WaveMamba, DENet, DWT, IWT


def checkpoint_state(path):
    obj = torch.load(path, map_location='cpu', weights_only=False)
    for key in ('params_ema', 'params', 'state_dict', 'model'):
        if isinstance(obj, dict) and isinstance(obj.get(key), dict):
            obj = obj[key]
            break
    return {k.removeprefix('module.'): v for k, v in obj.items()}


def compress_den(den, depth, keep):
    if not 1 <= keep < depth:
        raise ValueError('den_blocks phải nhỏ hơn số block của teacher.')
    selected = np.linspace(0, depth - 1, keep).round().astype(int).tolist()
    if keep == 1:
        selected = [0]
    for name in ('g1', 'g2', 'g3'):
        gp = getattr(den, name).gp
        retained = [gp[i] for i in selected] + [gp[depth]]
        getattr(den, name).gp = nn.Sequential(*retained)
    return selected


def teacher_and_student(cfg, checkpoint):
    WaveMamba, DENet, DWT, IWT = import_architecture(cfg['repo_root'])
    state = checkpoint_state(checkpoint)
    groups = {i: set() for i in (1, 2, 3)}
    for key in state:
        match = re.fullmatch(r'restoration_network\.DE\.g([123])\.gp\.(\d+)\.conv1\.weight', key)
        if match:
            groups[int(match[1])].add(int(match[2]))
    depths = {len(ids) for ids in groups.values()}
    if len(depths) != 1 or next(iter(depths)) not in (4, 6):
        raise ValueError(f'Không nhận diện được DEN checkpoint: {groups}')
    depth = next(iter(depths))
    if any(ids != set(range(depth)) for ids in groups.values()):
        raise ValueError('DEN block index không liên tục.')
    teacher = WaveMamba(in_chn=3, wf=16, n_l_blocks=[1, 2, 2, 4], ffn_scale=2.)
    teacher.restoration_network.DE = DENet(3, depth)
    teacher.load_state_dict(state, strict=True)
    student = copy.deepcopy(teacher)
    keep = int(cfg['den_blocks'])
    # Keep the group-terminal convolution and all FFT/attention/pre/post weights.
    # Group residual topology is preserved; only repeated expensive U-blocks are removed.
    selected = compress_den(student.restoration_network.DE, depth, keep)
    teacher = teacher.cuda().eval().requires_grad_(False)
    student = student.cuda().eval()
    with torch.inference_mode():
        x = torch.rand(1, 3, 64, 96, device='cuda')
        transform, inverse = DWT().cuda(), IWT().cuda()
        err = (inverse(transform(x)) - x).abs().max().item()
        assert err < 1e-6, f'DWT/IWT reconstruction lỗi: {err}'
        assert teacher(x).shape == student(x).shape == x.shape
        assert torch.isfinite(student(x)).all()
    print(f'DEN: {depth} → {keep} block/group; giữ index {selected}; DWT/IWT error={err:.2e}', flush=True)
    return teacher, student, depth


def load_pair(pair, max_side):
    images = []
    for key in ('hazy', 'clear'):
        image = cv2.imread(pair[key], cv2.IMREAD_COLOR)
        if image is None:
            raise OSError(f'Không đọc được {pair[key]}')
        images.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    x, y = images
    if (pair.get('dataset_name') == 'sots-its' and y.shape[:2] == (x.shape[0]+20, x.shape[1]+20)):
        # Standard SOTS indoor GT includes a 10-pixel frame absent from the hazy image.
        y = y[10:-10, 10:-10]
    if x.shape != y.shape:
        raise ValueError(f'Cặp ảnh khác shape: scene {pair["id"]}; {x.shape}, {y.shape}')
    h, w = x.shape[:2]
    if max_side and max(h, w) > max_side:
        scale = max_side / max(h, w)
        size = (round(w * scale), round(h * scale))
        x, y = [cv2.resize(a, size, interpolation=cv2.INTER_AREA) for a in (x, y)]
    return [torch.from_numpy(a.copy()).permute(2, 0, 1).float().div_(255) for a in (x, y)]


def pad32(x):
    h, w = x.shape[-2:]
    dh, dw = (-h) % 32, (-w) % 32
    return F.pad(x, (0, dw, 0, dh), mode='reflect' if dh < h and dw < w else 'replicate')


def final_output(y):
    return y[-1] if isinstance(y, (tuple, list)) else y


def metrics(pred, gt, border=4):
    a, b = [t.detach().float().clamp(0, 1).cpu().numpy().transpose(1, 2, 0) for t in (pred, gt)]
    if border:
        a, b = [v[border:-border, border:-border] for v in (a, b)]
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    psnr = float(-10 * np.log10(max(mse, 1e-12)))
    # Same 11x11 Gaussian window/sigma and valid-region averaging as BasicSR SSIM.
    ssim = float(structural_similarity(a, b, channel_axis=2, data_range=1.,
                                      gaussian_weights=True, sigma=1.5, use_sample_covariance=False))
    return psnr, ssim


@torch.inference_mode()
def quality(model, pairs, cfg, label, save_dir=None):
    model.eval()
    rows = []
    for pair in pairs:
        x, y = load_pair(pair, cfg['eval_max_side'])
        h, w = x.shape[-2:]
        pred = final_output(model(pad32(x[None].cuda())))[0, :, :h, :w].clamp(0, 1)
        psnr, ssim = metrics(pred, y, cfg['crop_border'])
        rows.append({'dataset': cfg.get('dataset_name', 'o-hazy'), 'model': label, 'id': pair['id'],
                     'height': h, 'width': w, 'PSNR': psnr, 'SSIM': ssim})
        if save_dir:
            save_dir = Path(save_dir)
            save_dir.mkdir(parents=True, exist_ok=True)
            a = (pred.permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)
            name = re.sub(r'[^\w.-]', '_', str(pair['id']))
            cv2.imwrite(str(save_dir / f'{name}_pred.png'), cv2.cvtColor(a, cv2.COLOR_RGB2BGR))
    return rows


@torch.inference_mode()
def latency(model, x, cfg):
    model.eval()
    for _ in range(cfg['warmup']):
        model(x)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(cfg['timing_repeats']):
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        start.record()
        result = model(x)
        end.record()
        end.synchronize()
        times.append(start.elapsed_time(end))
        del result
    return {'latency_ms': float(np.mean(times)), 'latency_std_ms': float(np.std(times)),
            'latency_p50_ms': float(np.percentile(times, 50)), 'latency_p95_ms': float(np.percentile(times, 95)),
            'FPS': 1000. / float(np.mean(times)), 'peak_allocated_GiB': torch.cuda.max_memory_allocated() / 2**30}


def freeze_bn(model):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


def update_ema(ema, model, decay):
    with torch.no_grad():
        for ep, p in zip(ema.parameters(), model.parameters()):
            ep.lerp_(p.detach(), 1 - decay)
        for eb, b in zip(ema.buffers(), model.buffers()):
            eb.copy_(b)

'''
RUNNER_SOURCE = r'''
"""Resume Phase 3, recover PSNR, and benchmark FP32 deployment plus estimated GFLOPs."""
import copy
import gc
import json
import math
import random
import subprocess
import sys
import time
from collections import defaultdict, OrderedDict
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

import phase3_core as core


def prepare(cfg):
    inputs = core.prepare_inputs(cfg)
    inputs['phase3_checkpoint'] = None
    inputs['phase3_sha256'] = None
    if not cfg.get('phase3_checkpoint'):
        core.dump_json(Path(cfg['work_root']) / 'inputs.json', inputs)
        print('Initialize student from compressed teacher; no O-HAZY student checkpoint.', flush=True)
        return
    path = Path(cfg['phase3_checkpoint']).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'Cần checkpoint Phase 3: {path}')
    saved = torch.load(path, map_location='cpu', weights_only=False)
    previous = saved.get('config', {})
    if int(previous.get('den_blocks', 1)) != 1:
        raise ValueError('V2 này tiếp tục từ Phase 3 có một block DEN/group.')
    name = core.normalize_dataset_name(cfg.get('dataset_name', 'o-hazy'))
    source_name = core.normalize_dataset_name(previous.get('dataset_name', 'o-hazy'))
    expected_source = 'reside6k' if name.startswith('sots-') and inputs['evaluation_only'] else name
    if source_name != expected_source:
        raise ValueError(f'Checkpoint student thuộc {source_name}, cần {expected_source}. Để trống để nén teacher.')
    if not inputs['evaluation_only'] and previous.get('val_ids', cfg['val_ids']) != cfg['val_ids']:
        raise ValueError('Giữ nguyên val_ids của Phase 3.')
    if name == 'o-hazy' and previous.get('eval_max_side', cfg['eval_max_side']) != cfg['eval_max_side']:
        raise ValueError('Giữ nguyên eval_max_side để so sánh với Phase 3.')
    if saved.get('teacher_sha256', inputs['checkpoint_sha256']) != inputs['checkpoint_sha256']:
        raise ValueError('Checkpoint teacher không khớp teacher của Phase 3.')
    inputs['phase3_checkpoint'] = str(path)
    inputs['phase3_sha256'] = core.sha256(path)
    core.dump_json(Path(cfg['work_root']) / 'inputs.json', inputs)
    print('Resume Phase 3:', path, flush=True)


def configure(cfg):
    random.seed(cfg['seed'])
    np.random.seed(cfg['seed'])
    torch.manual_seed(cfg['seed'])
    torch.cuda.manual_seed_all(cfg['seed'])
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core.cv2.setNumThreads(0)


def validate_inputs(cfg, inputs):
    if inputs['dataset_name'] != core.normalize_dataset_name(cfg.get('dataset_name', 'o-hazy')):
        raise ValueError('Dataset selector đã thay đổi; chạy prepare lại.')
    core.assert_disjoint(inputs['splits'])
    for split in inputs['splits'].values():
        for row in split:
            for key in ('hazy', 'clear'):
                if core.sha256(row[key]) != row[key+'_sha256']:
                    raise ValueError(f'Dataset đã thay đổi: {row[key]}')
    for key in ('checkpoint', 'phase3_checkpoint'):
        if not inputs.get(key):
            continue
        hash_key = 'checkpoint_sha256' if key == 'checkpoint' else 'phase3_sha256'
        if core.sha256(inputs[key]) != inputs[hash_key]:
            raise ValueError(f'Checkpoint đã thay đổi: {inputs[key]}')
    data, expected, eval_only = core.dataset_splits(cfg)
    if str(data) != inputs['data_root'] or eval_only != inputs['evaluation_only']:
        raise ValueError('Data root hoặc chế độ đã thay đổi; chạy prepare lại.')
    for name, rows in expected.items():
        keys = lambda values: [(r['id'], r['hazy'], r['clear']) for r in values]
        if keys(rows) != keys(inputs['splits'][name]):
            raise ValueError('Manifest split không khớp config hiện tại; chạy prepare lại.')


def restoration_loss(pred, gt, teacher, cfg):
    border = cfg['crop_border']
    if border:
        pred, gt, teacher = [v[..., border:-border, border:-border] for v in (pred, gt, teacher)]
    return (F.mse_loss(pred, gt) + cfg['l1_weight']*F.l1_loss(pred, gt)
            + cfg['kd_mse_weight']*F.mse_loss(pred, teacher))


def refine(cfg, inputs, teacher, student, output):
    """A short DEN stage followed by full-image joint training at evaluation resolution."""
    if inputs.get('evaluation_only'):
        return {'seconds': 0., 'steps': 0, 'initial_val_PSNR': None, 'best_val_PSNR': None,
                'mode': 'evaluation-only; không fine-tune trên test SOTS'}
    started = time.perf_counter()
    # Bounded LRU: RESIDE-6K must not materialize thousands of full FP32 images in RAM.
    cached = OrderedDict()
    capacity = int(cfg.get('max_cache_images', 40))
    if capacity < 1:
        raise ValueError('max_cache_images phải lớn hơn 0.')
    training_pairs = inputs['splits']['train']
    coarse = copy.deepcopy(student).eval().requires_grad_(False)
    coarse.restoration_network.DE = nn.Identity()
    @torch.no_grad()
    def cache_pair(row):
        key = row['hazy']
        if key not in cached:
            x, y = core.load_pair(row, cfg['train_max_side'])
            h, w = x.shape[-2:]
            xp = core.pad32(x[None].cuda())
            cached[key] = {'id': row['id'], 'x': x, 'y': y,
                           'coarse': coarse(xp)[0, :, :h, :w].cpu().contiguous(),
                           'teacher': teacher(xp)[0, :, :h, :w].cpu().contiguous()}
            if len(cached) > capacity:
                cached.popitem(last=False)
        cached.move_to_end(key)
        return cached[key]
    if len(training_pairs) <= capacity:
        for row in training_pairs:
            cache_pair(row)
            print(f'Cache train {row["id"]}', flush=True)
        coarse.cpu()
        teacher.cpu()
    else:
        print(f'Cache LRU {capacity} ảnh; lấy mẫu từ toàn bộ {len(training_pairs)} ảnh train.', flush=True)
    torch.cuda.empty_cache()
    ema = copy.deepcopy(student).eval().requires_grad_(False)
    initial = core.quality(ema, inputs['splits']['val'], cfg, 'initial_student_val')
    best = float(np.mean([r['PSNR'] for r in initial]))
    history = [{'stage': 'initial', 'step': 0, 'loss': 0., 'val_PSNR': best, 'elapsed_s': 0.}]
    path = output / 'fast_wdmamba_v2_best.pth'

    def save(step, score):
        torch.save({'params_ema': ema.state_dict(), 'config': cfg, 'step': step,
                    'val_PSNR': score, 'teacher_sha256': inputs['checkpoint_sha256'],
                    'phase3_sha256': inputs['phase3_sha256']}, path)

    save(0, best)  # A worse validation checkpoint never replaces the initial Phase 3.
    core.write_csv(output / 'initial_val_metrics.csv', initial)
    step = 0
    for stage, count in [('den_crop', cfg['crop_steps']), ('joint_full', cfg['full_steps'])]:
        if count <= 0:
            continue
        for name, p in student.named_parameters():
            p.requires_grad_(stage == 'joint_full' or name.startswith('restoration_network.DE.'))
        den, lfrn = [], []
        for name, p in student.named_parameters():
            if p.requires_grad:
                (den if name.startswith('restoration_network.DE.') else lfrn).append(p)
        groups = [{'params': den, 'lr': cfg['crop_lr'] if stage == 'den_crop' else cfg['full_den_lr']}]
        if lfrn:
            groups.append({'params': lfrn, 'lr': cfg['full_lfrn_lr']})
        opt = torch.optim.AdamW(groups, weight_decay=cfg['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(count, 1), eta_min=1e-7)
        student.train()
        core.freeze_bn(student)
        stale = 0
        for local_step in range(1, count+1):
            opt.zero_grad(set_to_none=True)
            batch_loss = 0.
            for _ in range(cfg['accum_steps']):
                row = cache_pair(random.choice(training_pairs))
                keys = ('coarse', 'y', 'teacher') if stage == 'den_crop' else ('x', 'y', 'teacher')
                images = [row[k] for k in keys]
                if stage == 'den_crop':
                    h, w = images[0].shape[-2:]
                    s = (min(cfg['crop_size'], h, w)//32)*32
                    if s < 32:
                        raise ValueError('Ảnh train phải có cạnh ngắn ít nhất 32 pixel.')
                    top, left = random.randrange(h-s+1), random.randrange(w-s+1)
                    images = [im[:, top:top+s, left:left+s] for im in images]
                    if random.random() < .5:
                        images = [im.flip(-1) for im in images]
                    if random.random() < .5:
                        images = [im.flip(-2) for im in images]
                x, gt, target = [im[None].cuda() for im in images]
                if stage == 'den_crop':
                    pred = student.restoration_network.DE(x)
                else:
                    # No crop/context mismatch: same full-image pad/crop as evaluator.
                    h, w = x.shape[-2:]
                    pred = core.final_output(student(core.pad32(x)))[..., :h, :w]
                loss = restoration_loss(pred, gt, target, cfg)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Loss không hữu hạn.')
                (loss/cfg['accum_steps']).backward()
                batch_loss += loss.item()/cfg['accum_steps']
            nn.utils.clip_grad_norm_([p for p in student.parameters() if p.requires_grad], 1., error_if_nonfinite=True)
            opt.step()
            scheduler.step()
            core.update_ema(ema, student, cfg['ema_decay'])
            step += 1
            if local_step % cfg['log_every'] == 0:
                print(f'{stage}: step={step}, loss={batch_loss:.6f}', flush=True)
            if local_step % cfg['val_every'] == 0 or local_step == count:
                rows = core.quality(ema, inputs['splits']['val'], cfg, 'V2_val')
                score = float(np.mean([r['PSNR'] for r in rows]))
                history.append({'stage': stage, 'step': step, 'loss': batch_loss, 'val_PSNR': score,
                                'elapsed_s': time.perf_counter()-started})
                core.write_csv(output / 'history_v2.csv', history)
                print(f'Validation={score:.4f}, best={best:.4f}', flush=True)
                if score > best:
                    best, stale = score, 0
                    save(step, score)
                else:
                    stale += 1
                if stale >= cfg['patience']:
                    print(f'Early stop stage {stage} bằng validation.', flush=True)
                    break
        # Start the next stage from the best EMA, not an overfit last iterate.
        student.load_state_dict(core.checkpoint_state(path), strict=True)
        ema.load_state_dict(student.state_dict(), strict=True)
    student.load_state_dict(core.checkpoint_state(path), strict=True)
    del ema, cached, coarse
    teacher.cpu()
    torch.cuda.empty_cache()
    return {'seconds': time.perf_counter()-started, 'steps': step,
            'initial_val_PSNR': history[0]['val_PSNR'], 'best_val_PSNR': best}


@torch.inference_mode(False)
@torch.no_grad()
def fuse_conv_bn(model):
    """Inference-only Conv2d/BatchNorm fusion; no graph or checkpoint key mutation during train."""
    fused = 0
    for module in list(model.modules()):
        if isinstance(module, nn.Sequential):
            names = list(module._modules)
            for left, right in zip(names, names[1:]):
                conv, bn = module._modules[left], module._modules[right]
                if isinstance(conv, nn.Conv2d) and isinstance(bn, nn.BatchNorm2d):
                    module._modules[left] = torch.nn.utils.fusion.fuse_conv_bn_eval(conv, bn)
                    module._modules[right] = nn.Identity()
                    fused += 1
    return fused


class GraphReplay(nn.Module):
    """One shape at a time; static output is consumed before the next replay."""
    def __init__(self, model):
        super().__init__()
        self.model = model.eval()
        self.release()

    def release(self):
        self.graph = self.static_x = self.static_y = None
        self.shape = None
        gc.collect()

    @torch.inference_mode()
    def forward(self, x):
        shape = (tuple(x.shape), x.dtype, x.device)
        if self.shape != shape:
            self.release()
            self.static_x = x.clone()
            stream = torch.cuda.Stream()
            stream.wait_stream(torch.cuda.current_stream())
            with torch.cuda.stream(stream):
                for _ in range(3):
                    self.model(self.static_x)
            torch.cuda.current_stream().wait_stream(stream)
            torch.cuda.synchronize()
            self.graph = torch.cuda.CUDAGraph()
            with torch.cuda.graph(self.graph):
                self.static_y = self.model(self.static_x)
            self.shape = shape
        self.static_x.copy_(x)
        self.graph.replay()
        return self.static_y


@torch.inference_mode(False)
@torch.no_grad()
def choose_deployment(model, cfg, inputs, output):
    """Decide runtime on validation only, before opening any test metric."""
    model.eval().cuda()
    if inputs['splits']['val']:
        selection_source = 'validation'
        x, _ = core.load_pair(inputs['splits']['val'][0], cfg['eval_max_side'])
        xp = core.pad32(x[None].cuda())
    else:
        selection_source = 'synthetic_256'
        # Evaluation-only datasets have no tuning split; select runtime on a synthetic shape probe.
        xp = torch.zeros(1, 3, 256, 256, device='cuda')
    reference = model(xp).clone()
    fused = fuse_conv_bn(model)
    error = (model(xp)-reference).abs().max().item()
    if error > cfg['fusion_max_error']:
        raise ValueError(f'Conv-BN fusion không đạt kiểm tra sai số: {error}')
    eager = core.latency(model, xp, cfg)
    info = {'selection_source': selection_source, 'fused_conv_bn': fused, 'fusion_max_abs_error': error, 'execution': 'fused_eager',
            'validation_eager_ms': eager['latency_ms']}
    graph = GraphReplay(model)
    chosen = model
    if cfg['try_cuda_graph']:
        try:
            y = graph(xp)
            graph_error = (y-model(xp)).abs().max().item()
            if graph_error > cfg['fusion_max_error']:
                raise ValueError(f'Graph output error={graph_error}')
            timing = core.latency(graph, xp, cfg)
            info.update(graph_max_abs_error=graph_error, validation_graph_ms=timing['latency_ms'])
            if timing['latency_ms'] < eager['latency_ms']:
                chosen = graph
                info['execution'] = 'fused_cuda_graph'
        except (RuntimeError, ValueError) as exc:
            info['graph_error'] = str(exc)
    if chosen is model:
        graph.release()
        # Verify fallback in the same ordinary-tensor mode used for later profiling.
        model(xp)
    core.dump_json(output / 'deployment_selection.json', info)
    del xp, reference
    print('Deployment selected:', info, flush=True)
    return chosen, info


def profile_gflops(model, x):
    """Analytical forward hooks; 1 multiply-add = 2 FLOPs, not GFLOP/s.

    Count Conv/Linear arithmetic exactly under dense zero-padding conventions.
    Functional SS2D projections are counted explicitly. Scan, FFT and Haar are
    supplemental estimates; activations, normalization, pooling and elementwise
    bookkeeping are not counted. This is not a complete hardware instruction count.
    """
    rows = defaultdict(float)
    handles = []

    def hook(module, args, result):
        a = args[0]
        name = module.__class__.__name__
        if isinstance(module, (nn.Conv1d, nn.Conv2d)):
            rows['counted_conv_linear'] += 2*result.numel()*(module.in_channels//module.groups)*math.prod(module.kernel_size)
        elif isinstance(module, nn.ConvTranspose2d):
            rows['counted_conv_linear'] += 2*a.numel()*(module.out_channels//module.groups)*math.prod(module.kernel_size)
        elif isinstance(module, nn.Linear):
            rows['counted_conv_linear'] += 2*a.numel()*module.out_features
        elif name == 'SS2D':
            b, h, w, _ = a.shape
            d, n, r = module.d_inner, module.d_state, module.dt_rank
            rows['counted_ss2d_einsum'] += 2*b*4*h*w*d*(2*r+2*n)
            rows['estimated_selective_scan'] += b*4*h*w*d*(8*n+2)
        elif name == 'fft_bench':
            b, c, h, w = a.shape
            rows['estimated_fft'] += 10*b*c*h*w*math.log2(h*w)
        elif name == 'DWTForward':
            # pytorch_wavelets functional Haar convolutions bypass nn.Conv hooks.
            rows['estimated_internal_haar'] += 4*a.numel()

    was_training = {m: m.training for m in model.modules()}
    try:
        model.eval()
        for module in model.modules():
            handles.append(module.register_forward_hook(hook))
        with torch.inference_mode():
            model(x)
    finally:
        for handle in handles:
            handle.remove()
        for module, training in was_training.items():
            module.training = training
    # Outer DWT/IWT are constructed inside forward and cannot be hooked persistently.
    if hasattr(model, 'restoration_network'):
        rows['estimated_outer_haar'] += 8*x.numel()
    counted = sum(v for k, v in rows.items() if k.startswith('counted_'))/1e9
    supplemental = sum(v for k, v in rows.items() if k.startswith('estimated_'))/1e9
    return {'counted_GFLOPs': counted, 'supplemental_GFLOPs': supplemental,
            'estimated_total_GFLOPs': counted+supplemental,
            **{k+'_GFLOPs': v/1e9 for k, v in rows.items()}}


def clear_graph(model):
    if isinstance(model, GraphReplay):
        model.release()
    torch.cuda.empty_cache()


def evaluate(model, cfg, inputs, label, execution, output):
    model.cuda().eval()
    physical = model.model if isinstance(model, GraphReplay) else model
    quality = core.quality(model, inputs['splits']['test'], cfg, label, output/'predictions'/label)
    clear_graph(model)
    timing, flops = [], []
    for row in inputs['splits']['test']:
        x, _ = core.load_pair(row, cfg['eval_max_side'])
        xp = core.pad32(x[None].cuda())
        h, w = xp.shape[-2:]
        # Eager arithmetic counter always profiles the physical model, never a replay.
        f = profile_gflops(physical, xp)
        flops.append({'dataset': cfg.get('dataset_name', 'o-hazy'), 'model': label, 'id': row['id'], 'height': h, 'width': w, **f})
        timing.append({'dataset': cfg.get('dataset_name', 'o-hazy'), 'model': label, 'id': row['id'], 'height': h, 'width': w,
                       **core.latency(model, xp, cfg)})
        clear_graph(model)
        del xp
    probe = torch.zeros(1, 3, 256, 256, device='cuda')
    f256 = profile_gflops(physical, probe)
    t256 = core.latency(model, probe, cfg)
    clear_graph(model)
    summary = {'dataset': cfg.get('dataset_name', 'o-hazy'), 'model': label, 'execution': execution, 'precision': 'FP32',
               'images': len(inputs['splits']['test']),
               'batch_size': 1, 'eval_max_side': cfg['eval_max_side'],
               'PSNR': float(np.mean([v['PSNR'] for v in quality])),
               'SSIM': float(np.mean([v['SSIM'] for v in quality])),
               'latency_ms': float(np.mean([v['latency_ms'] for v in timing])),
               'FPS': 1000/float(np.mean([v['latency_ms'] for v in timing])),
               'FPS_256': t256['FPS'], 'params_M': sum(p.numel() for p in physical.parameters())/1e6,
               'peak_allocated_GiB': max(v['peak_allocated_GiB'] for v in timing),
               'GFLOPs_256_counted': f256['counted_GFLOPs'],
               'GFLOPs_256_supplemental': f256['supplemental_GFLOPs'],
               'GFLOPs_256_estimated_total': f256['estimated_total_GFLOPs'],
               'GFLOPs_test_estimated_mean': float(np.mean([v['estimated_total_GFLOPs'] for v in flops]))}
    core.write_csv(output/f'{label}_metrics.csv', quality)
    core.write_csv(output/f'{label}_timing.csv', timing)
    core.write_csv(output/f'{label}_gflops.csv', flops)
    core.dump_json(output/f'{label}_256.json', {'latency': t256, 'gflops': f256})
    del probe
    model.cpu()
    torch.cuda.empty_cache()
    print(summary, flush=True)
    return summary


def restore_v2(cfg, inputs, student, output):
    """Load the already selected EMA checkpoint; never train or select by test metrics."""
    path = Path(cfg.get('v2_checkpoint') or output/'fast_wdmamba_v2_best.pth').expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'Không có checkpoint V2 đã train: {path}. Chế độ benchmark không chạy training.')
    saved = torch.load(path, map_location='cpu', weights_only=False)
    previous = saved.get('config', {})
    if saved.get('teacher_sha256') != inputs['checkpoint_sha256']:
        raise ValueError('Teacher của checkpoint V2 không khớp manifest.')
    if saved.get('phase3_sha256') != inputs.get('phase3_sha256'):
        raise ValueError('Baseline Phase 3 của checkpoint V2 không khớp manifest.')
    for key in ('dataset_name', 'den_blocks', 'eval_max_side', 'crop_border', 'val_ids'):
        if previous.get(key) != cfg.get(key):
            raise ValueError(f'Cấu hình checkpoint V2 không khớp {key}: {previous.get(key)} != {cfg.get(key)}')
    student.load_state_dict(core.checkpoint_state(path), strict=True)
    return {'mode': 'benchmark-only; không train lại', 'checkpoint': str(path),
            'checkpoint_sha256': core.sha256(path), 'selected_step': saved.get('step'),
            'best_val_PSNR': saved.get('val_PSNR')}


def run(cfg, benchmark_only=False):
    if not torch.cuda.is_available():
        raise RuntimeError('Cần GPU CUDA để train/benchmark. CPU chỉ dùng kiểm tra công thức.')
    configure(cfg)
    output = Path(cfg['output_root'])
    output.mkdir(parents=True, exist_ok=True)
    inputs = json.loads((Path(cfg['work_root'])/'inputs.json').read_text(encoding='utf-8'))
    validate_inputs(cfg, inputs)
    teacher, student, depth = core.teacher_and_student(cfg, inputs['checkpoint'])
    resumed = bool(inputs.get('phase3_checkpoint'))
    if resumed:
        student.load_state_dict(core.checkpoint_state(inputs['phase3_checkpoint']), strict=True)
    baseline_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
    training = (restore_v2(cfg, inputs, student, output) if benchmark_only
                else refine(cfg, inputs, teacher, student, output))
    student.eval()
    # Select runtime using validation; test metrics are still unopened.
    deployment, deploy_info = choose_deployment(copy.deepcopy(student), cfg, inputs, output)
    clear_graph(deployment)
    deployment.cpu()
    student.cpu()
    teacher.cpu()
    previous = copy.deepcopy(student)
    previous.load_state_dict(baseline_state, strict=True)
    rows = []
    baseline_label = 'Phase3' if resumed else 'Compressed_init'
    student_label = 'Student_eager' if inputs['evaluation_only'] else 'V2_eager'
    deploy_label = 'Student_deploy' if inputs['evaluation_only'] else 'V2_deploy'
    for model, name, execution in [(teacher, 'WDMamba', 'eager'), (previous, baseline_label, 'eager'),
                                   (student, student_label, 'eager'),
                                   (deployment, deploy_label, deploy_info['execution'])]:
        rows.append(evaluate(model, cfg, inputs, name, execution, output))
        # Keep completed model results if a later GPU operation fails.
        core.write_csv(output/'summary_partial.csv', rows)
    core.write_csv(output/'summary.csv', rows)
    base, old, new, deploy = rows
    if inputs['evaluation_only']:
        gates = {'deployment_quality': abs(deploy['PSNR']-new['PSNR']) <= cfg['deploy_psnr_tolerance_db']}
    else:
        gates = {'PSNR_gain_vs_baseline': new['PSNR']-old['PSNR'] >= cfg['min_psnr_gain_db'],
                 'FPS_gain_vs_baseline': deploy['FPS']/old['FPS'] >= cfg['min_fps_ratio'],
                 'teacher_quality': new['PSNR']-base['PSNR'] >= -cfg['psnr_tolerance_db'],
                 'deployment_quality': abs(deploy['PSNR']-new['PSNR']) <= cfg['deploy_psnr_tolerance_db']}
    status = ('EVALUATION_ONLY' if inputs['evaluation_only'] else 'PASS') if all(gates.values()) else 'REVIEW'
    metadata = {'status': status, 'gates': gates, 'baseline_model': baseline_label,
                'dataset': inputs['dataset_name'], 'evaluation_only': inputs['evaluation_only'],
                'delta_PSNR_vs_baseline': new['PSNR']-old['PSNR'],
                'delta_PSNR_vs_teacher': new['PSNR']-base['PSNR'],
                'deployment_speedup_vs_baseline': deploy['FPS']/old['FPS'],
                'eager_speedup_vs_baseline': new['FPS']/old['FPS'],
                'deployment': deploy_info, 'training': training,
                'gpu': torch.cuda.get_device_name(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
                'teacher_DEN_blocks': depth, 'phase3_sha256': inputs['phase3_sha256'],
                'teacher_sha256': inputs['checkpoint_sha256'],
                'GFLOPs_method': '1 MAC=2 FLOPs; Conv/Linear + SS2D projections counted; scan/FFT/Haar estimated; elementwise/norm/pooling excluded'}
    core.dump_json(output/'metadata.json', metadata)
    core.dump_json(output/'config.json', cfg)
    lines = [f"# Fast-WDMamba V2 — {inputs['dataset_name']}", '', f"Trạng thái: **{metadata['status']}**", '',
             '| Dataset | Model | Thực thi | PSNR | SSIM | FPS | FPS 256 | GFLOPs counted | GFLOPs bổ sung | GFLOPs tổng ước tính | Params M |',
             '|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|']
    for v in rows:
        lines.append(f"| {v['dataset']} | {v['model']} | {v['execution']} | {v['PSNR']:.4f} | {v['SSIM']:.5f} | {v['FPS']:.2f} | {v['FPS_256']:.2f} | {v['GFLOPs_256_counted']:.3f} | {v['GFLOPs_256_supplemental']:.3f} | {v['GFLOPs_256_estimated_total']:.3f} | {v['params_M']:.3f} |")
    lines += ['', f"ΔPSNR so {baseline_label}: {metadata['delta_PSNR_vs_baseline']:+.4f} dB; "
              f"tốc độ triển khai so baseline: {metadata['deployment_speedup_vs_baseline']:.3f}x.",
              f"ΔPSNR so teacher: {metadata['delta_PSNR_vs_teacher']:+.4f} dB.", '',
              f"Chế độ: {'chỉ đánh giá, không cập nhật trọng số' if inputs['evaluation_only'] else 'fine-tune train, chọn checkpoint bằng validation'}.",
              'GFLOPs gần như giữ nguyên giữa Phase 3 và V2; giảm kernel launch không đồng nghĩa giảm số phép tính.',
              'Đếm trên tensor đã pad, cùng hình dạng và FP32. GFLOPs là số phép tính/ảnh, không phải GFLOP/s.',
              'CSV tách counted và supplemental; không gọi supplemental là số FLOPs chính xác toàn mạng.',
              'Tất cả model được đo lại trong cùng phiên GPU. Không dùng FPS cũ từ file Phase 3 để tính speedup.',
              f"Đánh giá {len(inputs['splits']['test'])} ảnh test của {inputs['dataset_name']}; không dùng test để chọn checkpoint.",
              'Nếu REVIEW, mục tiêu tăng cả PSNR và FPS chưa được xác nhận; giữ nguyên số đo.']
    if inputs['dataset_name'] == 'o-hazy':
        lines += ['O-HAZY giữ test 41–45. Test Phase 3 đã được xem; đây là vòng phát triển tiếp.',
                  'Teacher O-HAZY đã học 1–40, gồm validation compression; điểm validation không độc lập với pretraining.']
    if inputs['dataset_name'] == 'i-haze' and Path(inputs['checkpoint']).name == 'Ohaze_1200_1600_27.22.pth':
        lines += ['I-HAZE dùng teacher O-HAZY làm khởi tạo chuyển miền; không có checkpoint I-HAZE gốc trong cấu hình mặc định.']
    if inputs['dataset_name'].startswith('sots-'):
        lines += ['SOTS là benchmark Indoor/Outdoor; ITS/OTS là các tập train riêng, có thể chỉ định bằng TRAIN_ROOT.',
                  'Nguồn teacher mặc định: RESIDE-6K. Cần kiểm tra giao scene với dữ liệu pretraining trước khi claim tổng quát hóa.']
    (output/'final_report.md').write_text('\n'.join(lines)+'\n', encoding='utf-8')
    print('\n'.join(lines), flush=True)


def benchmark(cfg):
    run(cfg, benchmark_only=True)


def auto(cfg):
    inputs = json.loads((Path(cfg['work_root'])/'inputs.json').read_text(encoding='utf-8'))
    checkpoint = Path(cfg.get('v2_checkpoint') or Path(cfg['output_root'])/'fast_wdmamba_v2_best.pth').expanduser()
    if not inputs['evaluation_only'] and (checkpoint.is_file() or cfg.get('v2_checkpoint')):
        print('AUTO: reuse saved V2 checkpoint; benchmark only, no training.', flush=True)
        benchmark(cfg)
    else:
        print('AUTO: run training/evaluation according to the dataset splits.', flush=True)
        run(cfg)


if __name__ == '__main__':
    cfg = json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
    {'prepare': prepare, 'run': run, 'benchmark': benchmark, 'auto': auto}[sys.argv[2]](cfg)

'''

import ast
ast.parse(CORE_SOURCE); ast.parse(RUNNER_SOURCE)
(WORK_ROOT/'phase3_core.py').write_text(CORE_SOURCE, encoding='utf-8')
RUNNER_PATH = WORK_ROOT/'refine_o_hazy.py'
RUNNER_PATH.write_text(RUNNER_SOURCE, encoding='utf-8')
def run_phase(phase):
    CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
    run_command([PYTHON, '-u', RUNNER_PATH, CONFIG_PATH, phase], cwd=REPO_ROOT, log_name=phase+'.log')
print('Runner v2.1: fusion fix + auto-resume; syntax PASS')

## 5. Chuẩn bị manifest và train/đánh giá V2

Giữ `EVAL_MAX_SIDE=768` để so Phase 3. `EVAL_ONLY=None` tự train nếu có train/val; SOTS chuẩn chỉ đánh giá. `EVAL_ONLY=False` yêu cầu train/val riêng. `TRAIN_ROOT` dành cho ITS/OTS tách khỏi test SOTS. I-HAZE giữ split trong repo; RESIDE-6K giữ train/test có sẵn và tách validation theo scene từ train nếu thiếu val. Nhiều ảnh hazy chung clear luôn nằm cùng split.

Cache RAM có giới hạn `MAX_CACHE_IMAGES`, lấy mẫu từ toàn bộ train; không chọn một tập con cố định cho RESIDE-6K. Crop tự giới hạn theo ảnh nhỏ. Nếu thiếu VRAM, giảm số bước full-image hoặc điều chỉnh cấu hình rồi ghi rõ protocol; notebook không tự giảm độ phân giải.

Mặc định `TRY_CUDA_GRAPH=False`: dùng fusion eager, tránh lỗi capture của CUDA/Mamba. Có thể bật để thử trên validation; chỉ dùng khi output gần eager và latency thấp hơn. Capture/setup không tính vào latency; copy vào static input có tính trong replay latency.

Mặc định `RUN_MODE='auto'`: nếu checkpoint `fast_wdmamba_v2_best.pth` còn trong output, notebook chỉ benchmark lại và kiểm tra nguồn checkpoint, không train lại. Chưa có checkpoint thì train/đánh giá theo dataset. Chọn `run` để chủ động train lại hoặc `benchmark` để bắt buộc dùng checkpoint đã có. Không thay bằng `EVAL_ONLY=True`, vì đó là chế độ đánh giá cross-dataset, không phải phục hồi lượt fine-tune.

In [ ]:
if RUN_MODE not in ('auto', 'run', 'benchmark'):
    raise ValueError('RUN_MODE phải là auto, run hoặc benchmark')
run_phase('prepare')
run_phase(RUN_MODE)

## 6. Bảng so sánh PSNR, FPS và GFLOPs

Mọi model đều FP32, batch 1, cùng dataset và evaluator. Baseline là `Phase3` nếu có student checkpoint, hoặc `Compressed_init` trước fine-tune. SOTS chỉ đánh giá xuất `Student_eager`/`Student_deploy`, trạng thái `EVALUATION_ONLY`; không dùng gate tăng PSNR để claim đã train cải tiến.
`V2_eager` chỉ thay trọng số; `V2_deploy` thêm tối ưu runtime. Gate kiểm tra: tăng PSNR ít nhất 0,10 dB so Phase 3, FPS triển khai tăng ít nhất 3%, giảm PSNR không quá 0,30 dB so teacher và PSNR deployment lệch eager không quá 0,01 dB.

**Lưu ý GFLOPs:** số counted không bao gồm activation, normalization, pooling, bias và phép elementwise. Selective scan/FFT/Haar là ước tính theo công thức công khai trong runner; tổng estimated không phải instruction count đầy đủ. ConvTranspose dùng quy ước MAC dày gồm vùng padding.

Test Phase 3 đã được xem để định hướng cải tiến này; đây là một vòng phát triển tiếp, không phải bộ holdout mới. Năm scene chưa đủ để khẳng định tổng quát hóa.

In [ ]:
from IPython.display import Markdown, display
import csv
display(Markdown((OUTPUT_ROOT/'final_report.md').read_text(encoding='utf-8')))
with (OUTPUT_ROOT/'summary.csv').open(encoding='utf-8') as f:
    for row in csv.DictReader(f):
        print(row['model'], 'counted GFLOPs 256:', row['GFLOPs_256_counted'],
              'supplemental:', row['GFLOPs_256_supplemental'],
              'test estimated mean:', row['GFLOPs_test_estimated_mean'])

## 7. Xuất checkpoint và toàn bộ số đo

`fast_wdmamba_v2_best.pth`, `history_v2.csv`, `summary.csv`, CSV GFLOPs/timing từng scene, ảnh dự đoán và metadata được lưu riêng, không ghi đè kết quả Phase 3.

Để dựng model: import `phase3_core`, gọi `teacher_and_student(config, teacher_path)` với `den_blocks=1`, rồi nạp `checkpoint_state(student_path)` vào student bằng `strict=True`. Inference FP32, pad bội 32 và crop lại. Fusion/CUDA Graph chỉ bật sau khi model ở eval; không nạp checkpoint eager vào model đã fusion.

In [ ]:
import shutil
for src in (WORK_ROOT/'phase3_core.py', RUNNER_PATH, WORK_ROOT/'inputs.json',
            WORK_ROOT/'split_manifest.csv', WORK_ROOT/'prepare.log', WORK_ROOT/(RUN_MODE+'.log')):
    if src.is_file(): shutil.copy2(src, OUTPUT_ROOT/src.name)
archive = shutil.make_archive(str(WORK_ROOT/('fast_wdmamba_v2_' + DATASET_NAME + '_results')), 'zip', OUTPUT_ROOT)
print('Kết quả:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass